In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt 
import matplotlib.patches as mpatches 
import csv 
import numpy as np 
import seaborn as sns 
from sklearn.metrics import root_mean_squared_error
import ast 

#### Style

In [ ]:
TASC_VColor         = "#57B8FF"  
TASC_permute_VColor = "#2176AE"  
SC_VColor           = "#7ED957"  
RSC_VColor          = "#FF9F40"  
CIM_VColor          = "#E255A1" 

XLABEL_FS = 14
YLABEL_FS = 14
YTICKS_FS = 14
XTICKS_FS = 14
LEGEND_FS = 14

In [ ]:
def process_tasc_df(df):
    list_cols = ["trueTarget", "pred_tasc"]
    for col in list_cols:
        df[col] = df[col].apply(ast.literal_eval)

    # Compute RMSE for TASC only
    df["rmse_pred_tasc"] = df.apply(
        lambda row: root_mean_squared_error(row["trueTarget"][row["T0"]:], row["pred_tasc"][row["T0"]:]),
        axis=1
    )

    return pd.DataFrame({
        "RMSE": df["rmse_pred_tasc"],
        "Method": ["TASC"] * len(df),
        "n": df["n"].tolist(),
        "T0": df["T0"].tolist(),
        "d": df["d"].tolist()
    })

def process_cim_df(df):
    list_cols = ["trueTarget", "pred_cim"]
    for col in list_cols:
        df[col] = df[col].apply(ast.literal_eval)

    df["rmse_pred_cim"] = df.apply(
        lambda row: root_mean_squared_error(row["trueTarget"][row["T0"]:], row["pred_cim"][row["T0"]:]),
        axis=1
    )
    
    return pd.DataFrame({
        "RMSE": df["rmse_pred_cim"],
        "Method": ["CIM"] * len(df), 
        "n": df["n"].tolist(),
        "T0": df["T0"].tolist(),
        "d": df["d"].tolist()
    })

def process_benchmark_df(df):
    list_cols = ["trueTarget", "pred_rsc", "pred_sc"]
    for col in list_cols:
        df[col] = df[col].apply(ast.literal_eval)

    df["rmse_pred_sc"] = df.apply(
        lambda row: root_mean_squared_error(row["trueTarget"][row["T0"]:], row["pred_sc"][row["T0"]:]),
        axis=1
    )
    df["rmse_pred_rsc"] = df.apply(
        lambda row: root_mean_squared_error(row["trueTarget"][row["T0"]:], row["pred_rsc"][row["T0"]:]),
        axis=1
    )

    return pd.DataFrame({
        "RMSE": np.concatenate([df["rmse_pred_sc"], df["rmse_pred_rsc"]]),
        "Method": (
            ["SC"] * len(df["rmse_pred_sc"]) +
            ["RSC"] * len(df["rmse_pred_rsc"])
        ),
        "n": df["n"].tolist() * 2,
        "T0": df["T0"].tolist() * 2,
        "d": df["d"].tolist() * 2
    })

files = [
    "resultLogsnba/nba_placebo_test_tasc_n24_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_tasc_n48_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_tasc_n96_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_tasc_n192_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_tasc_n384_d5_T096_aistats.csv",
]

optimal_benchmark_files_rsc_sc = [
    "resultLogsnba/nba_placebo_test_rsc_sc_n24_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_rsc_sc_n48_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_rsc_sc_n96_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_rsc_sc_n192_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_rsc_sc_n384_d5_T096_aistats.csv",
]

optimal_benchmark_files_cim = [
    "resultLogsnba/nba_placebo_test_cim_n24_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_cim_n48_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_cim_n96_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_cim_n192_d5_T096_aistats.csv",
    "resultLogsnba/nba_placebo_test_cim_n384_d5_T096_aistats.csv",
]

tasc_dfs = [process_tasc_df(pd.read_csv(f)) for f in files]
benchmark_dfs = [process_benchmark_df(pd.read_csv(f)) for f in optimal_benchmark_files_rsc_sc]
cim_dfs = [process_cim_df(pd.read_csv(f)) for f in optimal_benchmark_files_cim]

df_all = pd.concat(tasc_dfs + benchmark_dfs + cim_dfs, ignore_index=True)

T0 = df_all["T0"].unique().tolist()[0]
d = df_all["d"].unique().tolist()[0]

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_all,
    x="n",            
    y="RMSE",
    hue="Method",
    whis=[0, 95],
    palette={
        "TASC": TASC_VColor,
        "SC": SC_VColor,
        "RSC": RSC_VColor,
        "CIM": CIM_VColor
    },
)

plt.xlabel("Number of Donors (n)", fontsize=XLABEL_FS)
plt.ylabel("RMSE", fontsize=YLABEL_FS)
plt.ylim(0, 25)
plt.yticks(fontsize=YTICKS_FS)
plt.xticks(fontsize=XTICKS_FS)
plt.legend(title="Method", fontsize=LEGEND_FS, loc='upper left', title_fontsize=LEGEND_FS)
plt.tight_layout()
plt.minorticks_on()
plt.tick_params(axis="x", which="minor", bottom=False, top=False)   
plt.tick_params(axis="y", which="minor", left=True, right=False)     

plt.grid(axis='y', which="major", linestyle='-', linewidth=0.8, alpha=0.7)
plt.grid(axis='y', which="minor", linestyle='--', linewidth=0.5, alpha=0.4)

plt.show()

medians = df_all.groupby(["n", "Method"])["RMSE"].median().reset_index()
print(medians)

In [ ]:
def plot_horizon_plot_from_df_bench_smallSty(df_tasc, df_bench, df_cim, save_path=None):
    list_cols = ["trueTarget", "pred_sc", "pred_rsc"]
    for col in list_cols:
        df_bench[col] = df_bench[col].apply(ast.literal_eval)

    list_cols_tasc = ["trueTarget", "pred_tasc"]
    for col in list_cols_tasc:
        df_tasc[col] = df_tasc[col].apply(ast.literal_eval)

    list_cols_cim = ["trueTarget", "pred_cim"]
    for col in list_cols_cim:
        df_cim[col] = df_cim[col].apply(ast.literal_eval)

    block_size = 24   
    T = 192          
    rmse_data = []

    for _, row in df_tasc.iterrows():
        T0 = int(row["T0"])
        d = int(row["d"])
        n = int(row["n"])
        true_target = np.array(row["trueTarget"])
        tasc_pred = np.array(row["pred_tasc"])
        for start in range(T0, T, block_size):
            end = min(start + block_size, T)

            rmse_data.append({
                "RMSE": root_mean_squared_error(true_target[start:end], tasc_pred[start:end]),
                "Method": "TASC",
                "Block": (start - T0) // block_size + 1,
                "n": n
            })

    for _, row in df_bench.iterrows():
        T0 = int(row["T0"])
        d = int(row["d"])
        n = int(row["n"])
        true_target = np.array(row["trueTarget"])
        sc_pred = np.array(row["pred_sc"])
        rsc_pred = np.array(row["pred_rsc"])

        for start in range(T0, T, block_size):
            end = min(start + block_size, T)

            rmse_data.append({
                "RMSE": root_mean_squared_error(true_target[start:end], sc_pred[start:end]),
                "Method": "SC",
                "Block": (start - T0) // block_size + 1,
                "n": n
            })

            rmse_data.append({
                "RMSE": root_mean_squared_error(true_target[start:end], rsc_pred[start:end]),
                "Method": "RSC",
                "Block": (start - T0) // block_size + 1,
                "n": n
            })

    for _, row in df_cim.iterrows():
        T0 = int(row["T0"])
        d = int(row["d"])
        n = int(row["n"])
        true_target = np.array(row["trueTarget"])
        cim_pred = np.array(row["pred_cim"])
        for start in range(T0, T, block_size):
            end = min(start + block_size, T)

            rmse_data.append({
                "RMSE": root_mean_squared_error(true_target[start:end], cim_pred[start:end]),
                "Method": "CIM",
                "Block": (start - T0) // block_size + 1,
                "n": n
            })
    rmse_df = pd.DataFrame(rmse_data)

    plt.figure(figsize=(12, 6))
    ax = sns.boxplot(
        data=rmse_df,
        x="Block",
        y="RMSE",
        hue="Method",
        whis=[0, 95],
        palette={"TASC": TASC_VColor, "SC": SC_VColor, "RSC": RSC_VColor, "CIM": CIM_VColor},
    )

    num_blocks = len(rmse_df["Block"].unique())
    xtick_labels = ["1st half Q3", "2nd half Q3", "1st half Q4", "2nd half Q4"]
    ax.set_xticks(range(num_blocks))
    ax.set_xticklabels(xtick_labels, fontsize=14)

    plt.xlabel("Prediction Horizon (half quarter)", fontsize=14)
    plt.ylabel("RMSE", fontsize=14)
    plt.yticks(fontsize=14)
    plt.ylim(0, 25)
    plt.legend(title="Method", fontsize=12, loc='upper left', title_fontsize=14)

    plt.tight_layout()
    plt.minorticks_on()
    plt.grid(axis='y', which="major", linestyle='-', linewidth=0.8, alpha=0.7)
    plt.grid(axis='y', which="minor", linestyle='--', linewidth=0.5, alpha=0.4)
    plt.minorticks_on()
    plt.tick_params(axis="x", which="minor", bottom=False, top=False)   
    plt.tick_params(axis="y", which="minor", left=True, right=False)   

    plt.grid(axis='y', which="major", linestyle='-', linewidth=0.8, alpha=0.7)
    plt.grid(axis='y', which="minor", linestyle='--', linewidth=0.5, alpha=0.4)

    if save_path:
        plt.savefig(save_path)
        print(f"Saved horizon plot to {save_path}")
    plt.show()


df_tasc = pd.read_csv("resultLogsnba/nba_placebo_test_tasc_n192_d5_T096_aistats.csv")
df_bench = pd.read_csv("resultLogsnba/nba_placebo_test_rsc_sc_n192_d5_T096_aistats.csv")
df_cim = pd.read_csv("resultLogsnba/nba_placebo_test_cim_n192_d5_T096_aistats.csv")

plot_horizon_plot_from_df_bench_smallSty(df_tasc, df_bench, df_cim, save_path=None)

In [ ]:
df = pd.read_csv("resultLogsnba/nba_placebo_coeff_rsc_n96_d5_T077.csv")

list_cols = ["trueTarget", "pred_rsc_0_1", "pred_rsc_1", "pred_rsc_10", "pred_rsc_100", "pred_rsc_1000", "pred_rsc_10000", "pred_rsc_100000", "pred_rsc_1000000"]

for col in list_cols:
    df[col] = df[col].apply(ast.literal_eval)

pred_cols = ['pred_rsc_0_1', 'pred_rsc_1', 'pred_rsc_10', 'pred_rsc_100', 'pred_rsc_1000', 'pred_rsc_10000', 'pred_rsc_100000', 'pred_rsc_1000000']
for col in pred_cols: 
    df["rmse_"+col] = df.apply(
        lambda row: root_mean_squared_error(
            row["trueTarget"][row["T0"]:], row[col][row["T0"]:]
        ),
        axis=1
    )


df = pd.DataFrame({
    "RMSE": np.concatenate([df["rmse_pred_rsc_0_1"], df["rmse_pred_rsc_1"], df["rmse_pred_rsc_10"], df["rmse_pred_rsc_100"], df["rmse_pred_rsc_1000"], df["rmse_pred_rsc_10000"], df["rmse_pred_rsc_100000"], df["rmse_pred_rsc_1000000"]]),
    "Method": ["RIDGE-0.1"] * len(df["rmse_pred_rsc_0_1"]) + ["RIDGE-1.0"] * len(df["rmse_pred_rsc_1"]) +
                 ["RIDGE-10.0"] * len(df["rmse_pred_rsc_10"]) +
                 ["RIDGE-100.0"] * len(df["rmse_pred_rsc_100"]) +  
                 ["RIDGE-1000.0"] * len(df["rmse_pred_rsc_1000"]) +  
                 ["RIDGE-10000.0"] * len(df["rmse_pred_rsc_10000"]) +  
                 ["RIDGE-100000.0"] * len(df["rmse_pred_rsc_100000"])+
                 ["RIDGE-1000000.0"] * len(df["rmse_pred_rsc_1000000"])   
})

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df,
    x="Method", 
    y="RMSE",
    hue="Method",
    whis=[0, 95],
    palette={"RIDGE-0.1": "#1f77b4", "RIDGE-1.0": "#ff7f0e", "RIDGE-10.0": "#ff0eb3", "RIDGE-100.0": "#8BAE21", "RIDGE-1000.0": "#2FF02C", "RIDGE-10000.0": "#8899AA", "RIDGE-100000.0": "#668888", "RIDGE-1000000.0": "#229933"},
    legend=False,
)

plt.xlabel("Method-Lambda")
plt.ylim(bottom=0)               
plt.tight_layout()
plt.show()
medians = df.groupby(["Method"])["RMSE"].median().reset_index()
print(medians)
maximums = df.groupby(["Method"])["RMSE"].max().reset_index()
print(maximums)